In [ ]:
import pandas as pd
#creating data profile
def data_profile(df_analysis):
    return (
        df_analysis.isna()
        .sum()
        .to_frame("missing_count")
        .assign(
            missing_pct=lambda x: ((x["missing_count"] / len(df_analysis)) * 100).round(2),
            dtype=df_analysis.dtypes.astype(str),
            suspect_numeric_object=lambda x: (
                df_analysis.dtypes.astype(str).eq("object")
                & df_analysis.apply(
                    lambda col: (
                        pd.to_numeric(col.dropna(), errors="coerce").notna().mean() > 0.9
                        if col.dropna().shape[0] > 0
                        else False
                    )
                )
            ),
            unique_count=df_analysis.nunique(dropna=True),
            non_unique_count=lambda x: len(df_analysis) - x["unique_count"],
            non_unique_pct=lambda x: ((x["non_unique_count"] / len(df_analysis)) * 100).round(2),
            is_unique_key=lambda x: x["unique_count"] == len(df_analysis),
            has_duplicates=lambda x: x["unique_count"] < len(df_analysis),
            severity=lambda x: pd.cut(
                x["missing_pct"],
                bins=[-0.01, 0, 25, 50, 75, 100],
                labels=["None", "Low", "Moderate", "High", "Critical"],
                include_lowest=True
            )
        )
        .sort_values("missing_pct", ascending=False)
    )